# Portfolio inspection copy: CW2

Adapted from the completed coursework notebook. Embedded historical outputs and cloud metadata were removed for privacy and data redistribution control. The original notebook and its outputs are preserved unchanged outside this repository. Verified aggregate results are under `Project1/results`, `Project2/final_evaluation` or `CW2/results`.

Every original code cell is behind `PORTFOLIO_EXECUTE=False`, so Run All in the default state neither imports the ML stack nor trains, evaluates, installs packages, mounts Drive or writes files. Read code and saved result files first. See `../docs/REPRODUCIBILITY.md` before opting in. CW2 combines fitting and test scoring in one historical workflow; it requires all three opt-ins. No execution was used to regenerate the recorded metrics.


In [ ]:
PORTFOLIO_EXECUTE = False
RUN_TRAINING = False
RUN_FINAL_TEST = False
from pathlib import Path
REPO_ROOT = Path.cwd() if (Path.cwd()/"Project1").is_dir() else Path.cwd().parent
if PORTFOLIO_EXECUTE and not (RUN_TRAINING and RUN_FINAL_TEST):
    raise RuntimeError("CW2 full workflow requires explicit training and final-test opt-ins")


**M32366 Coursework 2 : CEREBRAL STROKE DATASET**




## 1 Setup and data access



### 1.1 Install the additional libraries

Colab already provides the core scientific stack. This cell adds the specialised imbalanced-learning, boosting and explanation libraries only when needed.

In [ ]:
if PORTFOLIO_EXECUTE:
    import sys, subprocess
    IN_COLAB = 'google.colab' in sys.modules
    if IN_COLAB:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'imbalanced-learn==0.14.2', 'xgboost==3.4.1', 'shap==0.52.0'])
    print('Runtime:', 'Google Colab' if IN_COLAB else 'Local validation')
else:
    print("Inspection mode: code not executed.")


### 1.2 Load Python and machine learning libraries

Libraries for data processing, modelling, evaluation and visualisation are imported separately from data loading so dependency errors are easier to diagnose.

In [ ]:
if PORTFOLIO_EXECUTE:
    import hashlib, json, math, platform, shutil, time, warnings, zipfile
    from dataclasses import dataclass
    from pathlib import Path
    import joblib, matplotlib, matplotlib.pyplot as plt, numpy as np, pandas as pd, seaborn as sns
    from IPython.display import Image, display
    from imblearn.ensemble import BalancedRandomForestClassifier
    from scipy.stats import binomtest
    from sklearn.base import BaseEstimator, TransformerMixin, clone
    from sklearn.compose import ColumnTransformer
    from sklearn.dummy import DummyClassifier
    from sklearn.experimental import enable_iterative_imputer
    from sklearn.impute import IterativeImputer, SimpleImputer
    from sklearn.inspection import permutation_importance
    from sklearn.linear_model import BayesianRidge, LogisticRegression
    from sklearn.metrics import (average_precision_score, brier_score_loss, confusion_matrix, f1_score, fbeta_score, precision_recall_curve, precision_score, recall_score, roc_auc_score, roc_curve)
    from sklearn.model_selection import (ParameterGrid, RandomizedSearchCV, RepeatedStratifiedKFold, StratifiedKFold, train_test_split)
    from sklearn.neighbors import KNeighborsRegressor
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
    from xgboost import XGBClassifier
    warnings.filterwarnings('ignore', category=FutureWarning)
    warnings.filterwarnings('ignore', category=UserWarning)
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    import sklearn
    import xgboost
    import imblearn
    import shap
    import numpy as np
    import pandas as pd

    print("Python:", platform.python_version())
    print("NumPy:", np.__version__)
    print("pandas:", pd.__version__)
    print("scikit-learn:", sklearn.__version__)
    print("XGBoost:", xgboost.__version__)
    print("imbalanced-learn:", imblearn.__version__)
    print("SHAP:", shap.__version__)
else:
    print("Inspection mode: code not executed.")


### 1.3 Set reproducible parameters

The random seed, test fraction, repeated validation design and bootstrap count are fixed before data inspection.

In [ ]:
if PORTFOLIO_EXECUTE:
    RANDOM_STATE = 42
    TEST_SIZE = 0.20
    CV_SPLITS = 5
    CV_REPEATS = 3
    BOOTSTRAP_REPS = 1000
    EXPECTED_SHA256 = '540986aebc2cc8a9a6c907df5ea4fdf93e8b5e3dabacdd727337363757b539aa'
    USE_GOOGLE_DRIVE = False
    DRIVE_FOLDER_NAME = 'ACTIVE - M32366 OIDS/CW2'
else:
    print("Inspection mode: code not executed.")


### 1.4 Mount Google Drive

Mount Google Drive to access the coursework dataset and project files.

In [ ]:
if PORTFOLIO_EXECUTE:
    print("Using repository-local files; Drive mounting disabled.")
else:
    print("Inspection mode: code not executed.")


### 1.5 Set the project workspace

Set the project workspace and create folders for saved analysis outputs.

In [ ]:
if PORTFOLIO_EXECUTE:
    project_root = REPO_ROOT / 'CW2'
    for relative in ['data/raw/extracted', 'evidence/figures', 'evidence/tables', 'evidence/models', 'submission']:
        (project_root / relative).mkdir(parents=True, exist_ok=True)
    print('Project workspace ready:', f'MyDrive/{DRIVE_FOLDER_NAME}' if IN_COLAB and USE_GOOGLE_DRIVE else DRIVE_FOLDER_NAME)
else:
    print("Inspection mode: code not executed.")


### 1.6 Locate the dataset

Locate the coursework dataset in the project folder.

In [ ]:
if PORTFOLIO_EXECUTE:
    dataset_path = project_root / 'dataset.csv'

    if not dataset_path.exists():
        raise FileNotFoundError(f'Dataset not found: {dataset_path}')

    print('Dataset found:', dataset_path.name)
else:
    print("Inspection mode: code not executed.")


### 1.7 Verify dataset integrity

The SHA-256 checksum verifies that the expected dataset is used throughout the analysis.

In [ ]:
if PORTFOLIO_EXECUTE:
    csv_hash = hashlib.sha256(dataset_path.read_bytes()).hexdigest()
    assert csv_hash == EXPECTED_SHA256, f'Unexpected dataset hash: {csv_hash}'
    print('Dataset integrity check passed:', csv_hash)
else:
    print("Inspection mode: code not executed.")


## 2 Context and methods

### Key assumptions

- `stroke` is an observed binary label without a documented prediction horizon.
- `Residence_type` is an address category, not proof of rural-clinic recruitment.
- PR-AUC is primary because the event rate is approximately 1.8%.
- EDA and model selection use development data; the fixed test set is reserved for final evaluation.

### 2.1 Configure outputs and plotting

Analysis outputs are saved in the project folder using a consistent visual style.

In [ ]:
if PORTFOLIO_EXECUTE:
    ROOT = project_root
    DATA_PATH = dataset_path
    FIGURE_DIR = ROOT / 'evidence' / 'figures'
    TABLE_DIR = ROOT / 'evidence' / 'tables'
    MODEL_DIR = ROOT / 'evidence' / 'models'
    PALETTE = {'navy':'#17365D','blue':'#2D6A9F','teal':'#4F8073','gold':'#C99428','orange':'#D97732','red':'#B44C43','grey':'#6B7280','light':'#E8EEF4'}
    sns.set_theme(style='whitegrid', context='notebook')
    plt.rcParams.update({'figure.figsize':(9.2,5.4),'figure.dpi':130,'savefig.dpi':220,'font.family':'DejaVu Sans','axes.titleweight':'bold','axes.titlesize':13,'axes.labelsize':10.5,'xtick.labelsize':9.5,'ytick.labelsize':9.5})
    print('Analysis outputs configured')
else:
    print("Inspection mode: code not executed.")


### 2.2 Declare raw and engineered feature sets

Keeping these lists visible makes every input to the preprocessing pipeline auditable.

In [ ]:
if PORTFOLIO_EXECUTE:
    RAW_NUMERIC = ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi']
    ENGINEERED_NUMERIC = RAW_NUMERIC + ['bmi_missing', 'smoking_missing', 'cardiometabolic_burden', 'age_hypertension', 'age_heart_disease', 'bmi_glucose_interaction']
    RAW_CATEGORICAL = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
    ENGINEERED_CATEGORICAL = RAW_CATEGORICAL + ['age_band']
    print('Raw features:', len(RAW_NUMERIC) + len(RAW_CATEGORICAL))
    print('Engineered feature fields:', len(ENGINEERED_NUMERIC) + len(ENGINEERED_CATEGORICAL))
else:
    print("Inspection mode: code not executed.")


### 2.3 Define reusable analysis functions

The following short cells expose the full preprocessing, metric and validation logic for inspection.

#### 2.3.1 Helper functions

These helper functions support figure saving, reproducible output storage, feature engineering and BMI imputation.

In [ ]:
if PORTFOLIO_EXECUTE:
    def save_figure(filename: str) -> None:
        plt.tight_layout()
        plt.savefig(FIGURE_DIR / filename, bbox_inches="tight", facecolor="white")
        plt.close()

    def json_ready(value):
        if isinstance(value, dict):
            return {str(k): json_ready(v) for k, v in value.items()}
        if isinstance(value, (list, tuple)):
            return [json_ready(v) for v in value]
        if isinstance(value, (np.integer,)):
            return int(value)
        if isinstance(value, (np.floating,)):
            return None if not np.isfinite(value) else float(value)
        if isinstance(value, np.ndarray):
            return json_ready(value.tolist())
        if isinstance(value, Path):
            return str(value)
        return value

    def write_json(path: Path, value) -> None:
        path.write_text(json.dumps(json_ready(value), indent=2), encoding="utf-8")

    def engineer_features(frame: pd.DataFrame) -> pd.DataFrame:
        result = frame.copy()
        result["bmi_missing"] = result["bmi"].isna().astype(int)
        result["smoking_missing"] = result["smoking_status"].isna().astype(int)
        result["cardiometabolic_burden"] = result["hypertension"] + result["heart_disease"]
        result["age_hypertension"] = result["age"] * result["hypertension"]
        result["age_heart_disease"] = result["age"] * result["heart_disease"]
        result["bmi_glucose_interaction"] = result["bmi"] * result["avg_glucose_level"]
        result["age_band"] = pd.cut(
            result["age"],
            bins=[-np.inf, 17, 39, 49, 59, 69, 79, np.inf],
            labels=["<18", "18-39", "40-49", "50-59", "60-69", "70-79", "80+"],
        ).astype(object)
        return result

    class BMIKNNImputer(BaseEstimator, TransformerMixin):
        """Training-fold KNN regression imputer for BMI using low-cost clinical fields."""

        def __init__(self, n_neighbors: int = 15, weights: str = "distance"):
            self.n_neighbors = n_neighbors
            self.weights = weights
            self.predictors = ["age", "avg_glucose_level", "hypertension", "heart_disease"]

        def fit(self, X, y=None):
            frame = pd.DataFrame(X).copy()
            observed = frame["bmi"].notna()
            self.fallback_ = float(frame.loc[observed, "bmi"].mean())
            self.scaler_ = StandardScaler().fit(frame.loc[observed, self.predictors])
            self.model_ = KNeighborsRegressor(
                n_neighbors=self.n_neighbors,
                weights=self.weights,
                algorithm="auto",
                n_jobs=-1,
            )
            self.model_.fit(
                self.scaler_.transform(frame.loc[observed, self.predictors]),
                frame.loc[observed, "bmi"],
            )
            return self

        def transform(self, X):
            frame = pd.DataFrame(X).copy()
            missing = frame["bmi"].isna()
            if missing.any():
                predicted = self.model_.predict(self.scaler_.transform(frame.loc[missing, self.predictors]))
                frame.loc[missing, "bmi"] = np.clip(predicted, 10.0, 98.0)
            frame["bmi"] = frame["bmi"].fillna(self.fallback_)
            return frame
else:
    print("Inspection mode: code not executed.")


#### 2.3.2 Preprocessing pipeline

This function builds the leakage-safe preprocessing pipeline for numeric and categorical features.

In [ ]:
if PORTFOLIO_EXECUTE:
    def make_preprocessor(imputation: str = "mean", engineered: bool = True) -> tuple[list, object]:
        numeric = ENGINEERED_NUMERIC if engineered else RAW_NUMERIC
        categorical = ENGINEERED_CATEGORICAL if engineered else RAW_CATEGORICAL
        if imputation == "iterative":
            numeric_imputer = IterativeImputer(
                estimator=BayesianRidge(),
                max_iter=10,
                random_state=RANDOM_STATE,
                skip_complete=True,
            )
        else:
            numeric_imputer = SimpleImputer(strategy="mean", add_indicator=False)
        preprocessor = ColumnTransformer(
            [
                (
                    "numeric",
                    Pipeline([("imputer", numeric_imputer), ("scale", StandardScaler())]),
                    numeric,
                ),
                (
                    "categorical",
                    Pipeline(
                        [
                            ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
                            ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
                        ]
                    ),
                    categorical,
                ),
            ],
            remainder="drop",
        )
        steps = [("engineer", FunctionLikeTransformer(engineered=engineered))]
        if imputation == "knn":
            steps.append(("knn_bmi", BMIKNNImputer()))
        return steps, preprocessor

    class FunctionLikeTransformer(BaseEstimator, TransformerMixin):
        def __init__(self, engineered: bool = True):
            self.engineered = engineered

        def fit(self, X, y=None):
            return self

        def transform(self, X):
            frame = pd.DataFrame(X).copy()
            return engineer_features(frame) if self.engineered else frame

    def make_pipeline(model, imputation: str = "mean", engineered: bool = True) -> Pipeline:
        steps, preprocessor = make_preprocessor(imputation=imputation, engineered=engineered)
        return Pipeline(steps + [("preprocess", preprocessor), ("model", model)])

    def safe_specificity(y_true, y_pred) -> float:
        tn, fp, _, _ = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        return tn / (tn + fp) if (tn + fp) else np.nan

    def metric_bundle(y_true, probabilities, threshold: float) -> dict:
        predictions = (np.asarray(probabilities) >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
        return {
            "threshold": float(threshold),
            "roc_auc": float(roc_auc_score(y_true, probabilities)),
            "pr_auc": float(average_precision_score(y_true, probabilities)),
            "recall": float(recall_score(y_true, predictions, zero_division=0)),
            "precision": float(precision_score(y_true, predictions, zero_division=0)),
            "f1": float(f1_score(y_true, predictions, zero_division=0)),
            "f2": float(fbeta_score(y_true, predictions, beta=2, zero_division=0)),
            "specificity": float(safe_specificity(y_true, predictions)),
            "brier": float(brier_score_loss(y_true, probabilities)),
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
        }
else:
    print("Inspection mode: code not executed.")


#### 2.3.3 Validation and threshold functions

These functions support F2-based threshold selection, repeated out-of-fold validation and bootstrap uncertainty estimation.

In [ ]:
if PORTFOLIO_EXECUTE:
    def select_f2_threshold(y_true, probabilities) -> tuple[float, pd.DataFrame]:
        thresholds = np.unique(np.quantile(probabilities, np.linspace(0.02, 0.98, 240)))
        rows = []
        for threshold in thresholds:
            metrics = metric_bundle(y_true, probabilities, float(threshold))
            rows.append(metrics)
        table = pd.DataFrame(rows)
        # In a screening context, retain at least 60% specificity to prevent an unbounded referral burden.
        feasible = table.loc[table["specificity"] >= 0.60]
        selection_pool = feasible if not feasible.empty else table
        selected = selection_pool.sort_values(["f2", "recall", "precision"], ascending=False).iloc[0]
        return float(selected["threshold"]), table

    def repeated_oof_probabilities(estimator, X, y, splits=CV_SPLITS, repeats=CV_REPEATS):
        splitter = RepeatedStratifiedKFold(
            n_splits=splits, n_repeats=repeats, random_state=RANDOM_STATE
        )
        probability_sum = np.zeros(len(y), dtype=float)
        prediction_count = np.zeros(len(y), dtype=int)
        fold_rows = []
        start = time.time()
        for fold_number, (train_index, valid_index) in enumerate(splitter.split(X, y), start=1):
            fitted = clone(estimator)
            fitted.fit(X.iloc[train_index], y.iloc[train_index])
            probabilities = fitted.predict_proba(X.iloc[valid_index])[:, 1]
            probability_sum[valid_index] += probabilities
            prediction_count[valid_index] += 1
            fold_rows.append(
                {
                    "fold": fold_number,
                    "repeat": (fold_number - 1) // splits + 1,
                    "fold_within_repeat": (fold_number - 1) % splits + 1,
                    "validation_records": len(valid_index),
                    "validation_strokes": int(y.iloc[valid_index].sum()),
                    "roc_auc": roc_auc_score(y.iloc[valid_index], probabilities),
                    "pr_auc": average_precision_score(y.iloc[valid_index], probabilities),
                }
            )
        if not np.all(prediction_count == repeats):
            raise RuntimeError("Repeated OOF coverage failed")
        return probability_sum / prediction_count, pd.DataFrame(fold_rows), time.time() - start

    def bootstrap_metric_intervals(y_true, probabilities, threshold, reps=BOOTSTRAP_REPS):
        rng = np.random.default_rng(RANDOM_STATE)
        y_array = np.asarray(y_true)
        p_array = np.asarray(probabilities)
        metrics = ["roc_auc", "pr_auc", "recall", "precision", "f1", "f2", "specificity", "brier"]
        samples = {metric: [] for metric in metrics}
        for _ in range(reps):
            while True:
                indices = rng.integers(0, len(y_array), len(y_array))
                if np.unique(y_array[indices]).size == 2:
                    break
            result = metric_bundle(y_array[indices], p_array[indices], threshold)
            for metric in metrics:
                samples[metric].append(result[metric])
        point = metric_bundle(y_array, p_array, threshold)
        return {
            metric: {
                "estimate": point[metric],
                "lower_95": float(np.quantile(samples[metric], 0.025)),
                "upper_95": float(np.quantile(samples[metric], 0.975)),
            }
            for metric in metrics
        }
else:
    print("Inspection mode: code not executed.")


#### 2.3.4 Statistical comparison and subgroup functions

These functions support paired model comparison, confidence intervals and subgroup evaluation.

In [ ]:
if PORTFOLIO_EXECUTE:
    def paired_bootstrap_difference(y_true, first, second, metric="pr_auc", reps=BOOTSTRAP_REPS):
        rng = np.random.default_rng(RANDOM_STATE + 7)
        y_array = np.asarray(y_true)
        first = np.asarray(first)
        second = np.asarray(second)
        differences = []
        scorer = average_precision_score if metric == "pr_auc" else roc_auc_score
        for _ in range(reps):
            while True:
                indices = rng.integers(0, len(y_array), len(y_array))
                if np.unique(y_array[indices]).size == 2:
                    break
            differences.append(scorer(y_array[indices], first[indices]) - scorer(y_array[indices], second[indices]))
        observed = scorer(y_array, first) - scorer(y_array, second)
        return {
            "metric": metric,
            "difference": float(observed),
            "lower_95": float(np.quantile(differences, 0.025)),
            "upper_95": float(np.quantile(differences, 0.975)),
            "two_sided_bootstrap_p": float(2 * min(np.mean(np.asarray(differences) <= 0), np.mean(np.asarray(differences) >= 0))),
        }

    def wilson_interval(successes: int, total: int, z: float = 1.96) -> tuple[float, float]:
        if total == 0:
            return np.nan, np.nan
        proportion = successes / total
        denominator = 1 + z**2 / total
        centre = (proportion + z**2 / (2 * total)) / denominator
        half = z * math.sqrt(proportion * (1 - proportion) / total + z**2 / (4 * total**2)) / denominator
        return centre - half, centre + half

    def subgroup_table(frame, y_true, probabilities, threshold, column):
        predictions = (np.asarray(probabilities) >= threshold).astype(int)
        rows = []
        for group in pd.Series(frame[column]).fillna("Missing").unique():
            mask = pd.Series(frame[column]).fillna("Missing").eq(group).to_numpy()
            group_y = np.asarray(y_true)[mask]
            group_pred = predictions[mask]
            group_prob = np.asarray(probabilities)[mask]
            positives = int(group_y.sum())
            tp = int(((group_y == 1) & (group_pred == 1)).sum())
            recall = tp / positives if positives else np.nan
            lower, upper = wilson_interval(tp, positives)
            rows.append(
                {
                    "attribute": column,
                    "group": str(group),
                    "records": int(mask.sum()),
                    "stroke_events": positives,
                    "predicted_positive": int(group_pred.sum()),
                    "true_positive": tp,
                    "false_negative": int(((group_y == 1) & (group_pred == 0)).sum()),
                    "recall": recall,
                    "recall_lower_95": lower,
                    "recall_upper_95": upper,
                    "precision": precision_score(group_y, group_pred, zero_division=0),
                    "specificity": safe_specificity(group_y, group_pred),
                    "pr_auc": average_precision_score(group_y, group_prob) if positives else np.nan,
                }
            )
        return pd.DataFrame(rows)
else:
    print("Inspection mode: code not executed.")


## 3 Data audit, exploratory analysis and principal experiments

Each stage is split into focused cells. Outputs appear beside the code that creates them.

### 3.1 Start analysis timer

Record the analysis start time for the final runtime summary.

In [ ]:
if PORTFOLIO_EXECUTE:
    run_started = time.time()
else:
    print("Inspection mode: code not executed.")


### 3.2 Validate the source and lock the split

Confirm schema, uniqueness and prevalence before creating the stratified development and test partitions.

In [ ]:
if PORTFOLIO_EXECUTE:
    print("1/10 Loading and validating the fixed source dataset")

    data = pd.read_csv(DATA_PATH)

    expected_columns = [
        "id", "gender", "age", "hypertension", "heart_disease", "ever_married",
        "work_type", "Residence_type", "avg_glucose_level", "bmi", "smoking_status", "stroke",
    ]

    if data.columns.tolist() != expected_columns or data.shape != (43400, 12):
        raise ValueError("Dataset schema or shape differs from the audited source")

    if data["id"].duplicated().any() or data.duplicated().any():
        raise ValueError("Dataset uniqueness check failed")

    X = data.drop(columns=["stroke", "id"])

    y = data["stroke"].astype(int)

    X_dev, X_test, y_dev, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
    )

    split_summary = pd.DataFrame(
        [
            {"partition": "Development", "records": len(y_dev), "strokes": int(y_dev.sum()), "stroke_rate": y_dev.mean()},
            {"partition": "Locked test", "records": len(y_test), "strokes": int(y_test.sum()), "stroke_rate": y_test.mean()},
        ]
    )

    split_summary.to_csv(TABLE_DIR / "split_summary.csv", index=False)

    integrity_rows = [
        {"check": "Records", "result": len(data), "status": "Pass"},
        {"check": "Columns", "result": data.shape[1], "status": "Pass"},
        {"check": "Duplicate IDs", "result": int(data.id.duplicated().sum()), "status": "Pass"},
        {"check": "Exact duplicate rows", "result": int(data.duplicated().sum()), "status": "Pass"},
        {"check": "Stroke events", "result": int(y.sum()), "status": "Observed"},
        {"check": "BMI missing", "result": int(data.bmi.isna().sum()), "status": "Requires treatment"},
        {"check": "Smoking status missing", "result": int(data.smoking_status.isna().sum()), "status": "Requires treatment"},
        {"check": "BMI above 80", "result": int((data.bmi > 80).sum()), "status": "Sensitivity check"},
    ]

    pd.DataFrame(integrity_rows).to_csv(TABLE_DIR / "data_integrity_summary.csv", index=False)
else:
    print("Inspection mode: code not executed.")


### 3.3.1 Examine class imbalance

Quantify the development-set class distribution and recorded stroke prevalence before modelling.

In [ ]:
if PORTFOLIO_EXECUTE:
    print("2/10 Producing development-only exploratory analysis")

    dev = X_dev.copy()

    dev["stroke"] = y_dev.values

    class_counts = dev.stroke.value_counts().sort_index()

    plt.figure()

    ax = sns.barplot(x=["No recorded stroke", "Recorded stroke"], y=class_counts.values, color=PALETTE["blue"])

    ax.set_ylim(0, class_counts.max() * 1.14)

    ax.bar_label(ax.containers[0], labels=[f"{v:,}\n({v/len(dev):.2%})" for v in class_counts.values], padding=3, fontsize=9)

    ax.set(title="Severe class imbalance in the development cohort", xlabel="Outcome", ylabel="Records")

    save_figure("fig01_class_balance.png")

    display(Image(filename=str(FIGURE_DIR / "fig01_class_balance.png")))
else:
    print("Inspection mode: code not executed.")


### 3.3.2 Examine missing data

Quantify development-set missingness to identify variables requiring explicit preprocessing.

In [ ]:
if PORTFOLIO_EXECUTE:
    missing_by_outcome = []

    for field in ["bmi", "smoking_status"]:
        for label, group in dev.groupby("stroke"):
            missing_by_outcome.append(
                {"field": field, "stroke": label, "missing_pct": group[field].isna().mean() * 100, "records": len(group)}
            )

    missing_by_outcome = pd.DataFrame(missing_by_outcome)

    plt.figure()

    ax = sns.barplot(
        data=missing_by_outcome.assign(
            field=lambda d: d.field.map({"bmi": "BMI", "smoking_status": "Smoking status"}),
            outcome=lambda d: d.stroke.map({0: "No stroke", 1: "Stroke"}),
        ),
        x="field", y="missing_pct", hue="outcome", palette=[PALETTE["blue"], PALETTE["red"]],
    )

    ax.set(title="Missingness differs by outcome", xlabel="Field", ylabel="Missing records (%)")

    ax.legend(title="Outcome")

    save_figure("fig02_missingness_by_outcome.png")

    display(Image(filename=str(FIGURE_DIR / "fig02_missingness_by_outcome.png")))
else:
    print("Inspection mode: code not executed.")


### 3.3.3 Compare age distributions by stroke status

Compare age distributions between recorded stroke and non-stroke cases while highlighting their substantial overlap.

In [ ]:
if PORTFOLIO_EXECUTE:
    plt.figure()

    sample_for_violin = pd.concat(
        [dev.loc[dev.stroke == 0].sample(6000, random_state=RANDOM_STATE), dev.loc[dev.stroke == 1]],
        ignore_index=True,
    )

    ax = sns.violinplot(
        data=sample_for_violin, x="stroke", y="age", hue="stroke",
        palette=[PALETTE["blue"], PALETTE["red"]], inner="quartile", legend=False, cut=0,
    )

    ax.set(title="Age separates the classes but distributions still overlap", xlabel="Outcome", ylabel="Age (years)")

    ax.set_xticks([0, 1], ["No recorded stroke", "Recorded stroke"])

    save_figure("fig03_age_violin.png")

    display(Image(filename=str(FIGURE_DIR / "fig03_age_violin.png")))
else:
    print("Inspection mode: code not executed.")


### 3.3.4 Compare BMI distributions by stroke status

Compare BMI distributions between recorded stroke and non-stroke cases, highlighting the long upper tail and substantial overlap.

In [ ]:
if PORTFOLIO_EXECUTE:
    plt.figure()

    ax = sns.boxenplot(
        data=sample_for_violin, x="stroke", y="bmi", hue="stroke",
        palette=[PALETTE["teal"], PALETTE["gold"]], legend=False, showfliers=True,
    )

    ax.set(title="BMI has a long upper tail and substantial class overlap", xlabel="Outcome", ylabel="BMI (kg/m2)")

    ax.set_xticks([0, 1], ["No recorded stroke", "Recorded stroke"])

    save_figure("fig04_bmi_boxen.png")

    display(Image(filename=str(FIGURE_DIR / "fig04_bmi_boxen.png")))
else:
    print("Inspection mode: code not executed.")


### 3.3.5 Compare glucose distributions by stroke status

Compare average glucose distributions between recorded stroke and non-stroke cases to assess class separation and distributional overlap.

In [ ]:
if PORTFOLIO_EXECUTE:
    plt.figure()

    ax = sns.kdeplot(
        data=sample_for_violin, x="avg_glucose_level", hue="stroke", common_norm=False,
        palette=[PALETTE["blue"], PALETTE["red"]], fill=False, linewidth=2,
    )

    ax.set(title="Stroke records show a heavier high-glucose tail", xlabel="Average glucose level", ylabel="Density")

    legend = ax.get_legend()

    if legend:
        legend.set_title("Outcome")
        for text, label in zip(legend.texts, ["No stroke", "Stroke"]):
            text.set_text(label)

    save_figure("fig05_glucose_distribution.png")

    display(Image(filename=str(FIGURE_DIR / "fig05_glucose_distribution.png")))
else:
    print("Inspection mode: code not executed.")


### 3.3.6 Examine feature associations
Examine associations among selected numerical and binary development features using a Spearman/phi correlation heatmap.

In [ ]:
if PORTFOLIO_EXECUTE:
    association = engineer_features(dev.drop(columns="stroke"))

    association["stroke"] = dev.stroke.values

    association_numeric = pd.get_dummies(
        association[
            ["age", "hypertension", "heart_disease", "avg_glucose_level", "bmi",
             "bmi_missing", "smoking_missing", "cardiometabolic_burden", "Residence_type", "stroke"]
        ],
        columns=["Residence_type"], dummy_na=False, dtype=float,
    )

    correlation = association_numeric.corr(method="spearman")

    plt.figure(figsize=(9.5, 7.2))

    mask = np.triu(np.ones_like(correlation, dtype=bool), k=1)

    ax = sns.heatmap(
        correlation, mask=mask, cmap="vlag", center=0, vmin=-1, vmax=1,
        annot=True, fmt=".2f", annot_kws={"size": 7.5}, square=False,
        cbar_kws={"label": "Spearman / phi association"},
    )

    ax.set_title("Mixed numeric and binary associations in development data")

    save_figure("fig06_association_heatmap.png")

    display(Image(filename=str(FIGURE_DIR / "fig06_association_heatmap.png")))
else:
    print("Inspection mode: code not executed.")


### 3.3.7 Compare stroke prevalence by residence type

Compare observed stroke prevalence between rural and urban development records with Wilson confidence intervals and event counts.

In [ ]:
if PORTFOLIO_EXECUTE:
    residence = dev.groupby("Residence_type").stroke.agg(records="size", strokes="sum", rate="mean").reset_index()

    intervals = [wilson_interval(int(row.strokes), int(row.records)) for row in residence.itertuples()]

    residence["lower"] = [x[0] for x in intervals]

    residence["upper"] = [x[1] for x in intervals]

    plt.figure()

    x_positions = np.arange(len(residence))

    rates = residence.rate.to_numpy() * 100

    errors = np.vstack([(residence.rate - residence.lower) * 100, (residence.upper - residence.rate) * 100])

    plt.bar(x_positions, rates, color=PALETTE["teal"], width=0.62)

    plt.errorbar(x_positions, rates, yerr=errors, fmt="none", color=PALETTE["navy"], capsize=5, linewidth=1.6)

    plt.xticks(x_positions, residence.Residence_type)

    plt.ylabel("Observed stroke prevalence (%)")

    plt.xlabel("Residence type")

    plt.title("Rural and urban prevalence estimates overlap")

    for x, rate, strokes in zip(x_positions, rates, residence.strokes):
        plt.text(x, rate + errors[1, x] + 0.06, f"{rate:.2f}%\n({int(strokes)} events)", ha="center", va="bottom", fontsize=9)

    save_figure("fig07_residence_prevalence_ci.png")

    display(Image(filename=str(FIGURE_DIR / "fig07_residence_prevalence_ci.png")))
else:
    print("Inspection mode: code not executed.")


### 3.3.8 Compare cardiovascular conditions by stroke status

Compare the prevalence of recorded hypertension and heart disease between stroke and non-stroke cases in the development set.

In [ ]:
if PORTFOLIO_EXECUTE:
    risk_rows = []

    for risk in ["hypertension", "heart_disease"]:
        for stroke_label, group in dev.groupby("stroke"):
            risk_rows.append({"risk": risk, "outcome": stroke_label, "prevalence": group[risk].mean() * 100})

    risk_table = pd.DataFrame(risk_rows)

    plt.figure()

    ax = sns.barplot(
        data=risk_table.assign(
            risk=lambda d: d.risk.map({"hypertension": "Hypertension", "heart_disease": "Heart disease"}),
            outcome=lambda d: d.outcome.map({0: "No stroke", 1: "Stroke"}),
        ),
        x="risk", y="prevalence", hue="outcome", palette=[PALETTE["blue"], PALETTE["red"]],
    )

    ax.set(title="Cardiovascular conditions are more common in stroke records", xlabel="Recorded condition", ylabel="Condition prevalence (%)")

    ax.legend(title="Outcome")

    save_figure("fig08_risk_factor_prevalence.png")

    display(Image(filename=str(FIGURE_DIR / "fig08_risk_factor_prevalence.png")))
else:
    print("Inspection mode: code not executed.")


### 3.3.9 Save the development EDA summary

Consolidate key development-set EDA statistics and save them as structured evidence for reproducibility and reporting.

In [ ]:
if PORTFOLIO_EXECUTE:
    eda_summary = {
        "development_records": len(dev),
        "development_strokes": int(dev.stroke.sum()),
        "development_prevalence_pct": dev.stroke.mean() * 100,
        "missing_by_outcome": missing_by_outcome.to_dict("records"),
        "residence": residence.to_dict("records"),
        "age_by_outcome": dev.groupby("stroke").age.describe().reset_index().to_dict("records"),
        "numeric_by_outcome": dev.groupby("stroke")[["age", "bmi", "avg_glucose_level"]].median().reset_index().to_dict("records"),
    }

    write_json(TABLE_DIR / "eda_summary.json", eda_summary)
else:
    print("Inspection mode: code not executed.")


### 3.4 Compare BMI imputation

Hold the model and validation design constant while comparing mean, iterative and KNN strategies.

In [ ]:
if PORTFOLIO_EXECUTE:
    print("3/10 Comparing leakage-safe BMI imputation strategies")

    imputation_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    imputation_rows = []

    for imputation in ["mean", "knn", "iterative"]:
        estimator = make_pipeline(
            LogisticRegression(max_iter=2500, class_weight="balanced", C=0.1, random_state=RANDOM_STATE),
            imputation=imputation,
            engineered=True,
        )
        fold_number = 0
        start = time.time()
        for train_index, valid_index in imputation_cv.split(X_dev, y_dev):
            fold_number += 1
            fitted = clone(estimator).fit(X_dev.iloc[train_index], y_dev.iloc[train_index])
            probabilities = fitted.predict_proba(X_dev.iloc[valid_index])[:, 1]
            imputation_rows.append(
                {
                    "imputation": imputation,
                    "fold": fold_number,
                    "roc_auc": roc_auc_score(y_dev.iloc[valid_index], probabilities),
                    "pr_auc": average_precision_score(y_dev.iloc[valid_index], probabilities),
                }
            )
        elapsed = time.time() - start
        for row in imputation_rows:
            if row["imputation"] == imputation:
                row["total_seconds"] = elapsed

    imputation_results = pd.DataFrame(imputation_rows)

    imputation_results.to_csv(TABLE_DIR / "imputation_comparison_folds.csv", index=False)

    imputation_summary = imputation_results.groupby("imputation")[["roc_auc", "pr_auc", "total_seconds"]].agg(["mean", "std"]).reset_index()

    imputation_summary.columns = ["_".join(str(v) for v in col if v).rstrip("_") for col in imputation_summary.columns]

    imputation_summary.to_csv(TABLE_DIR / "imputation_comparison_summary.csv", index=False)

    best_imputation = imputation_summary.sort_values("pr_auc_mean", ascending=False).iloc[0]["imputation"]

    print("Selected imputation:", best_imputation)
else:
    print("Inspection mode: code not executed.")


### 3.5.1 Configure candidate model tuning

Define the prespecified candidate models, hyperparameter search spaces and stratified development-only cross-validation procedure using PR-AUC for selection.

In [ ]:
if PORTFOLIO_EXECUTE:
    print("4/10 Tuning candidate models on development data")

    imbalance_ratio = (len(y_dev) - y_dev.sum()) / y_dev.sum()

    base_models = {
        "Logistic regression": LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE),
        "Random forest": __import__("sklearn.ensemble", fromlist=["RandomForestClassifier"]).RandomForestClassifier(
            n_estimators=250, class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1
        ),
        "XGBoost": XGBClassifier(
            objective="binary:logistic", eval_metric="logloss", tree_method="hist",
            n_estimators=300, scale_pos_weight=float(imbalance_ratio), random_state=RANDOM_STATE,
            n_jobs=-1, verbosity=0,
        ),
        "Balanced random forest": BalancedRandomForestClassifier(
            n_estimators=250, random_state=RANDOM_STATE, n_jobs=-1, sampling_strategy="all", replacement=True
        ),
    }

    search_spaces = {
        "Logistic regression": {"model__C": np.logspace(-3, 1, 9).tolist()},
        "Random forest": {
            "model__max_depth": [5, 8, 12, None],
            "model__min_samples_leaf": [2, 5, 10, 20],
            "model__max_features": ["sqrt", 0.6, 1.0],
        },
        "XGBoost": {
            "model__max_depth": [2, 3, 4, 5],
            "model__learning_rate": [0.02, 0.04, 0.07, 0.1],
            "model__min_child_weight": [1, 3, 6, 10],
            "model__subsample": [0.7, 0.85, 1.0],
            "model__colsample_bytree": [0.7, 0.85, 1.0],
            "model__reg_lambda": [1, 5, 10, 20],
        },
        "Balanced random forest": {
            "model__max_depth": [5, 8, 12, None],
            "model__min_samples_leaf": [2, 5, 10, 20],
            "model__max_features": ["sqrt", 0.6, 1.0],
        },
    }

    search_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

    tuned_models = {}

    search_rows = []
else:
    print("Inspection mode: code not executed.")


### 3.5.2 Execute candidate model tuning

Run development-only randomized hyperparameter searches for each candidate model and save the resulting PR-AUC scores and selected configurations.

In [ ]:
if PORTFOLIO_EXECUTE:
    for name, model in base_models.items():
        pipeline = make_pipeline(model, imputation=str(best_imputation), engineered=True)
        space = search_spaces[name]
        combinations = len(list(ParameterGrid(space)))
        n_iter = min(16, combinations)
        search = RandomizedSearchCV(
            pipeline, param_distributions=space, n_iter=n_iter, scoring="average_precision",
            cv=search_cv, random_state=RANDOM_STATE, n_jobs=1, refit=True,
            return_train_score=False,
        )
        start = time.time()
        search.fit(X_dev, y_dev)
        elapsed = time.time() - start
        tuned_models[name] = search.best_estimator_
        frame = pd.DataFrame(search.cv_results_)
        for _, row in frame.iterrows():
            search_rows.append(
                {
                    "model": name,
                    "mean_cv_pr_auc": row["mean_test_score"],
                    "std_cv_pr_auc": row["std_test_score"],
                    "rank": row["rank_test_score"],
                    "parameters": json.dumps(row["params"], sort_keys=True),
                    "search_seconds": elapsed,
                }
            )
        print(name, search.best_score_, search.best_params_)

    search_results = pd.DataFrame(search_rows).sort_values(["model", "rank"])

    search_results.to_csv(TABLE_DIR / "hyperparameter_search.csv", index=False)
else:
    print("Inspection mode: code not executed.")


### 3.6.1 Generate repeated out-of-fold predictions

Generate repeated stratified out-of-fold development predictions for each tuned candidate model to support unbiased model comparison and threshold selection.

In [ ]:
if PORTFOLIO_EXECUTE:
    print("5/10 Generating repeated out-of-fold model evidence")

    model_oof = {}

    fold_frames = []

    timing_rows = []

    for name, estimator in tuned_models.items():
        probabilities, folds, elapsed = repeated_oof_probabilities(estimator, X_dev, y_dev)
        model_oof[name] = probabilities
        folds.insert(0, "model", name)
        fold_frames.append(folds)
        timing_rows.append({"model": name, "repeated_cv_seconds": elapsed})
        print(name, average_precision_score(y_dev, probabilities), roc_auc_score(y_dev, probabilities))

    age_only = Pipeline(
        [
            ("preprocess", ColumnTransformer([("age", StandardScaler(), ["age"])])),
            ("model", LogisticRegression(max_iter=1500, class_weight="balanced", random_state=RANDOM_STATE)),
        ]
    )

    dummy = Pipeline(
        [
            ("preprocess", ColumnTransformer([("age", SimpleImputer(strategy="mean"), ["age"])])),
            ("model", DummyClassifier(strategy="prior")),
        ]
    )

    for name, estimator in [("Age-only logistic", age_only), ("Prevalence dummy", dummy)]:
        probabilities, folds, elapsed = repeated_oof_probabilities(estimator, X_dev, y_dev)
        model_oof[name] = probabilities
        folds.insert(0, "model", name)
        fold_frames.append(folds)
        timing_rows.append({"model": name, "repeated_cv_seconds": elapsed})

    model_oof["All-negative rule"] = np.zeros(len(y_dev), dtype=float)

    timing_rows.append({"model": "All-negative rule", "repeated_cv_seconds": 0.0})

    base_names = ["Logistic regression", "Random forest", "XGBoost", "Balanced random forest"]

    meta_features = np.column_stack([model_oof[name] for name in base_names])

    meta_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE + 11)

    meta_sum = np.zeros(len(y_dev))

    meta_count = np.zeros(len(y_dev), dtype=int)

    for train_index, valid_index in meta_cv.split(meta_features, y_dev):
        meta = LogisticRegression(max_iter=1500, C=0.3, random_state=RANDOM_STATE)
        meta.fit(meta_features[train_index], y_dev.iloc[train_index])
        meta_sum[valid_index] += meta.predict_proba(meta_features[valid_index])[:, 1]
        meta_count[valid_index] += 1

    model_oof["Stacked ensemble"] = meta_sum / meta_count

    repeated_folds = pd.concat(fold_frames, ignore_index=True)

    repeated_folds.to_csv(TABLE_DIR / "repeated_cv_fold_results.csv", index=False)

    pd.DataFrame(timing_rows).to_csv(TABLE_DIR / "model_timing.csv", index=False)

    oof_rows = []

    threshold_tables = {}

    model_thresholds = {}
else:
    print("Inspection mode: code not executed.")


### 3.6.2 Select the model and development thresholds

Evaluate repeated out-of-fold predictions, select model-specific F2 thresholds, and freeze the primary model using development PR-AUC before accessing the test set.

In [ ]:
if PORTFOLIO_EXECUTE:
    for name, probabilities in model_oof.items():
        if name == "All-negative rule":
            threshold = float(np.nextafter(0.0, 1.0))
            threshold_table = pd.DataFrame([metric_bundle(y_dev, probabilities, threshold)])
        else:
            threshold, threshold_table = select_f2_threshold(y_dev, probabilities)
        model_thresholds[name] = threshold
        threshold_tables[name] = threshold_table.assign(model=name)
        oof_rows.append({"model": name, **metric_bundle(y_dev, probabilities, threshold)})

    oof_metrics = pd.DataFrame(oof_rows).sort_values("pr_auc", ascending=False)

    oof_metrics.to_csv(TABLE_DIR / "oof_model_metrics.csv", index=False)

    pd.concat(threshold_tables.values(), ignore_index=True).to_csv(TABLE_DIR / "threshold_sensitivity.csv", index=False)

    candidate_oof = oof_metrics.loc[
        ~oof_metrics.model.isin(["Prevalence dummy", "All-negative rule", "Age-only logistic"])
    ]

    selected_model_name = str(candidate_oof.iloc[0].model)

    selected_threshold = float(model_thresholds[selected_model_name])

    print("Selected model before test access:", selected_model_name, "threshold", selected_threshold)

    plt.figure(figsize=(9.5, 5.8))

    plot_metrics = oof_metrics.melt(id_vars="model", value_vars=["roc_auc", "pr_auc"], var_name="metric", value_name="score")

    ax = sns.barplot(data=plot_metrics, y="model", x="score", hue="metric", palette=[PALETTE["blue"], PALETTE["gold"]])

    ax.set(title="Repeated out-of-fold discrimination", xlabel="Score", ylabel="Model")

    ax.legend(title="Metric", labels=["ROC-AUC", "PR-AUC"])

    save_figure("fig09_repeated_cv_model_comparison.png")

    display(Image(filename=str(FIGURE_DIR / "fig09_repeated_cv_model_comparison.png")))
else:
    print("Inspection mode: code not executed.")


### 3.6.3 Examine the development threshold trade-off

Visualise precision, recall and F2 across development thresholds and mark the prespecified selected threshold.

In [ ]:
if PORTFOLIO_EXECUTE:
    plt.figure()

    selected_threshold_table = threshold_tables[selected_model_name]

    sns.lineplot(data=selected_threshold_table, x="threshold", y="recall", label="Recall", color=PALETTE["red"])

    sns.lineplot(data=selected_threshold_table, x="threshold", y="precision", label="Precision", color=PALETTE["blue"])

    sns.lineplot(data=selected_threshold_table, x="threshold", y="f2", label="F2", color=PALETTE["gold"])

    plt.axvline(selected_threshold, color=PALETTE["navy"], linestyle="--", label=f"Selected {selected_threshold:.3f}")

    plt.title(f"Development threshold trade-off for {selected_model_name}")

    plt.xlabel("Decision threshold")

    plt.ylabel("Score")

    plt.legend()

    save_figure("fig10_threshold_sensitivity.png")

    display(Image(filename=str(FIGURE_DIR / "fig10_threshold_sensitivity.png")))
else:
    print("Inspection mode: code not executed.")


### 3.7 Open the untouched test once

Fit the selected development pipelines and evaluate the locked test at frozen thresholds.

In [ ]:
if PORTFOLIO_EXECUTE:
    print("6/10 Freezing models and accessing the untouched test set once")

    test_probabilities = {}

    fitted_models = {}

    for name, estimator in tuned_models.items():
        fitted = clone(estimator).fit(X_dev, y_dev)
        fitted_models[name] = fitted
        test_probabilities[name] = fitted.predict_proba(X_test)[:, 1]
        joblib.dump(fitted, MODEL_DIR / f"{name.lower().replace(' ', '_')}.joblib")

    fitted_age = clone(age_only).fit(X_dev, y_dev)

    fitted_dummy = clone(dummy).fit(X_dev, y_dev)

    test_probabilities["Age-only logistic"] = fitted_age.predict_proba(X_test)[:, 1]

    test_probabilities["Prevalence dummy"] = fitted_dummy.predict_proba(X_test)[:, 1]

    test_probabilities["All-negative rule"] = np.zeros(len(y_test), dtype=float)

    meta_model = LogisticRegression(max_iter=1500, C=0.3, random_state=RANDOM_STATE)

    meta_model.fit(meta_features, y_dev)

    stacked_test_features = np.column_stack([test_probabilities[name] for name in base_names])

    test_probabilities["Stacked ensemble"] = meta_model.predict_proba(stacked_test_features)[:, 1]

    joblib.dump(meta_model, MODEL_DIR / "stacked_meta_model.joblib")

    test_rows = []

    for name, probabilities in test_probabilities.items():
        test_rows.append({"model": name, **metric_bundle(y_test, probabilities, model_thresholds[name])})

    test_metrics = pd.DataFrame(test_rows).sort_values("pr_auc", ascending=False)

    test_metrics.to_csv(TABLE_DIR / "untouched_test_metrics.csv", index=False)

    selected_probabilities = test_probabilities[selected_model_name]

    selected_test_metrics = metric_bundle(y_test, selected_probabilities, selected_threshold)

    test_intervals = bootstrap_metric_intervals(y_test, selected_probabilities, selected_threshold)

    write_json(TABLE_DIR / "selected_test_confidence_intervals.json", test_intervals)
else:
    print("Inspection mode: code not executed.")


### 3.8 Compare paired predictions

Quantify uncertainty in XGBoost-versus-logistic differences on the same held-out observations.

In [ ]:
if PORTFOLIO_EXECUTE:
    print("7/10 Performing paired statistical comparisons")

    comparator_name = "Logistic regression"

    paired_pr = paired_bootstrap_difference(
        y_test, selected_probabilities, test_probabilities[comparator_name], metric="pr_auc"
    )

    paired_roc = paired_bootstrap_difference(
        y_test, selected_probabilities, test_probabilities[comparator_name], metric="roc_auc"
    )

    selected_predictions = (selected_probabilities >= selected_threshold).astype(int)

    comparator_predictions = (test_probabilities[comparator_name] >= model_thresholds[comparator_name]).astype(int)

    selected_correct = selected_predictions == y_test.to_numpy()

    comparator_correct = comparator_predictions == y_test.to_numpy()

    b = int((selected_correct & ~comparator_correct).sum())

    c = int((~selected_correct & comparator_correct).sum())

    mcnemar_p = float(binomtest(min(b, c), n=b + c, p=0.5, alternative="two-sided").pvalue) if b + c else 1.0

    statistics = {
        "selected_model": selected_model_name,
        "comparator": comparator_name,
        "paired_bootstrap_pr_auc": paired_pr,
        "paired_bootstrap_roc_auc": paired_roc,
        "mcnemar": {"selected_only_correct": b, "comparator_only_correct": c, "exact_p": mcnemar_p},
    }

    write_json(TABLE_DIR / "statistical_comparison.json", statistics)
else:
    print("Inspection mode: code not executed.")


### 3.9.1 Plot ROC and precision–recall curves

Compare held-out discrimination across candidate models using ROC and precision–recall curves.

In [ ]:
if PORTFOLIO_EXECUTE:
    print("8/10 Producing final curves, calibration and subgroup audit")

    plt.figure(figsize=(10.5, 4.6))

    ax1 = plt.subplot(1, 2, 1)

    ax2 = plt.subplot(1, 2, 2)

    curve_models = ["Age-only logistic", "Logistic regression", "XGBoost", "Balanced random forest", "Stacked ensemble"]

    colors = [PALETTE["grey"], PALETTE["blue"], PALETTE["orange"], PALETTE["teal"], PALETTE["navy"]]

    for name, color in zip(curve_models, colors):
        probabilities = test_probabilities[name]
        fpr, tpr, _ = roc_curve(y_test, probabilities)
        precision, recall, _ = precision_recall_curve(y_test, probabilities)
        ax1.plot(fpr, tpr, color=color, linewidth=1.8, label=f"{name} ({roc_auc_score(y_test, probabilities):.3f})")
        ax2.plot(recall, precision, color=color, linewidth=1.8, label=f"{name} ({average_precision_score(y_test, probabilities):.3f})")

    ax1.plot([0, 1], [0, 1], "--", color="#999999", linewidth=1)

    ax1.set(title="ROC curves", xlabel="False-positive rate", ylabel="True-positive rate")

    ax2.axhline(y_test.mean(), linestyle="--", color="#999999", linewidth=1, label=f"Prevalence ({y_test.mean():.3f})")

    ax2.set(title="Precision-recall curves", xlabel="Recall", ylabel="Precision")

    ax1.legend(fontsize=7.5, loc="lower right")

    ax2.legend(fontsize=7.5, loc="upper right")

    save_figure("fig11_roc_pr_curves.png")

    display(Image(filename=str(FIGURE_DIR / "fig11_roc_pr_curves.png")))
else:
    print("Inspection mode: code not executed.")


### 3.9.2 Review the held-out confusion matrix

Summarise the selected model's false positives, false negatives, true negatives and true positives on the untouched test set.

In [ ]:
if PORTFOLIO_EXECUTE:
    matrix = confusion_matrix(y_test, selected_predictions, labels=[0, 1])

    plt.figure(figsize=(6.4, 5.3))

    ax = sns.heatmap(matrix, annot=True, fmt=",d", cmap="Blues", cbar=False, linewidths=1, linecolor="white")

    ax.set(title=f"Untouched-test confusion matrix: {selected_model_name}", xlabel="Predicted label", ylabel="Observed label")

    ax.set_xticklabels(["No stroke", "Stroke"])

    ax.set_yticklabels(["No stroke", "Stroke"], rotation=0)

    save_figure("fig12_confusion_matrix.png")

    display(Image(filename=str(FIGURE_DIR / "fig12_confusion_matrix.png")))
else:
    print("Inspection mode: code not executed.")


### 3.9.3 Assess probability calibration

Compare predicted probabilities with observed stroke prevalence on the untouched test set using a calibration curve.

In [ ]:
if PORTFOLIO_EXECUTE:
    plt.figure()

    for name, color in [("Logistic regression", PALETTE["blue"]), (selected_model_name, PALETTE["navy"])]:
        probabilities = test_probabilities[name]
        bins = pd.qcut(probabilities, q=10, duplicates="drop")
        calibration = pd.DataFrame({"probability": probabilities, "outcome": y_test.to_numpy(), "bin": bins}).groupby("bin", observed=False).agg(
            mean_predicted=("probability", "mean"), observed_rate=("outcome", "mean"), records=("outcome", "size")
        ).reset_index()
        plt.plot(calibration.mean_predicted, calibration.observed_rate, marker="o", linewidth=2, color=color, label=name)

    plt.plot([0, max(0.16, selected_probabilities.max())], [0, max(0.16, selected_probabilities.max())], "--", color="#888888", label="Ideal")

    plt.title("Untouched-test calibration by probability decile")

    plt.xlabel("Mean predicted probability")

    plt.ylabel("Observed stroke rate")

    plt.legend()

    save_figure("fig13_calibration_curve.png")

    display(Image(filename=str(FIGURE_DIR / "fig13_calibration_curve.png")))
else:
    print("Inspection mode: code not executed.")


### 3.9.4 Audit subgroup performance

Evaluate held-out performance across residence type, gender, age group and BMI availability, retaining subgroup event counts for interpretation.

In [ ]:
if PORTFOLIO_EXECUTE:
    subgroup_frames = []

    X_test_audit = X_test.copy()

    X_test_audit["age_group"] = pd.cut(
        X_test_audit.age, [-np.inf, 39, 59, 69, 79, np.inf], labels=["<40", "40-59", "60-69", "70-79", "80+"]
    ).astype(object)

    X_test_audit["bmi_availability"] = np.where(X_test_audit.bmi.isna(), "BMI missing", "BMI observed")

    for column in ["Residence_type", "gender", "age_group", "bmi_availability"]:
        subgroup_frames.append(subgroup_table(X_test_audit, y_test, selected_probabilities, selected_threshold, column))

    subgroup_results = pd.concat(subgroup_frames, ignore_index=True)

    subgroup_results.to_csv(TABLE_DIR / "subgroup_audit.csv", index=False)
else:
    print("Inspection mode: code not executed.")


### 3.10.1 Calculate interpretable evidence

Use permutation importance as predictive association evidence, not causal evidence.

In [ ]:
if PORTFOLIO_EXECUTE:
    print("9/10 Calculating interpretable feature evidence")

    importance_estimator_name = selected_model_name if selected_model_name in fitted_models else "XGBoost"

    importance_estimator = fitted_models[importance_estimator_name]

    importance_sample = X_test.sample(min(2500, len(X_test)), random_state=RANDOM_STATE)

    importance_target = y_test.loc[importance_sample.index]

    importance = permutation_importance(
        importance_estimator,
        importance_sample,
        importance_target,
        scoring="average_precision",
        n_repeats=10,
        random_state=RANDOM_STATE,
        n_jobs=1,
    )

    importance_table = pd.DataFrame(
        {
            "feature": X_test.columns,
            "importance_mean": importance.importances_mean,
            "importance_sd": importance.importances_std,
        }
    ).sort_values("importance_mean", ascending=False)

    importance_table.to_csv(TABLE_DIR / "permutation_importance.csv", index=False)

    plt.figure(figsize=(8.5, 5.8))

    top_importance = importance_table.head(10).sort_values("importance_mean")

    plt.barh(top_importance.feature, top_importance.importance_mean, xerr=top_importance.importance_sd, color=PALETTE["teal"], capsize=3)

    plt.axvline(0, color="#666666", linewidth=1)

    plt.title(f"Permutation importance for {importance_estimator_name}")

    plt.xlabel("Decrease in average precision after permutation")

    plt.ylabel("Original feature")

    save_figure("fig14_permutation_importance.png")

    display(Image(filename=str(FIGURE_DIR / "fig14_permutation_importance.png")))
else:
    print("Inspection mode: code not executed.")


### 3.11.1 Save experiment and environment records

Save the experiment log and software environment metadata used to reproduce the analysis.

In [ ]:
if PORTFOLIO_EXECUTE:
    print("10/10 Saving experiment log and final evidence summary")

    experiment_log = []

    for _, row in imputation_summary.iterrows():
        experiment_log.append(
            {
                "stage": "Imputation",
                "variant": row.imputation,
                "validation_pr_auc": row.pr_auc_mean,
                "validation_roc_auc": row.roc_auc_mean,
                "decision": "Retained" if row.imputation == best_imputation else "Rejected",
                "reason": "Highest leakage-safe mean PR-AUC" if row.imputation == best_imputation else "Lower mean PR-AUC or disproportionate computation",
            }
        )

    for _, row in oof_metrics.iterrows():
        experiment_log.append(
            {
                "stage": "Model",
                "variant": row.model,
                "validation_pr_auc": row.pr_auc,
                "validation_roc_auc": row.roc_auc,
                "decision": "Selected" if row.model == selected_model_name else "Comparator/rejected",
                "reason": "Highest eligible repeated-OOF PR-AUC before test access" if row.model == selected_model_name else "Benchmark or lower repeated-OOF PR-AUC",
            }
        )

    pd.DataFrame(experiment_log).to_csv(TABLE_DIR / "experiment_log.csv", index=False)

    environment = {
        "python": sys.version,
        "platform": platform.platform(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": __import__("sklearn").__version__,
        "xgboost": __import__("xgboost").__version__,
        "shap": __import__("shap").__version__,
        "imbalanced_learn": __import__("imblearn").__version__,
        "random_state": RANDOM_STATE,
    }

    write_json(TABLE_DIR / "environment_versions.json", environment)
else:
    print("Inspection mode: code not executed.")


### 3.11.2 Save the final evidence summary

Consolidate the dataset, split, selected model, threshold, development metrics, held-out results and statistical comparisons into the final machine-readable summary.

In [ ]:
if PORTFOLIO_EXECUTE:
    final_summary = {
        "dataset": {
    "sha256": csv_hash,
            "records": len(data),
            "stroke_events": int(y.sum()),
            "stroke_prevalence_pct": y.mean() * 100,
        },
        "split": split_summary.to_dict("records"),
        "selected_imputation": best_imputation,
        "selected_model": selected_model_name,
        "selected_threshold": selected_threshold,
        "development_oof_metrics": oof_metrics.loc[oof_metrics.model == selected_model_name].iloc[0].to_dict(),
        "untouched_test_metrics": selected_test_metrics,
        "test_confidence_intervals": test_intervals,
        "statistical_comparison": statistics,
        "runtime_seconds": time.time() - run_started,
    }

    write_json(TABLE_DIR / "final_summary.json", final_summary)

    print(json.dumps(json_ready(final_summary), indent=2))
else:
    print("Inspection mode: code not executed.")


## 4 Prespecified sensitivity calibration and interpretability

These checks challenge the primary analysis without changing the already frozen model-selection rule.

### 4.1 Load supplementary libraries

These imports support fold-safe oversampling, dimension reduction, calibration and SHAP.

In [ ]:
if PORTFOLIO_EXECUTE:
    import shap
    from imblearn.over_sampling import RandomOverSampler
    from imblearn.pipeline import Pipeline as ImbPipeline
    from sklearn.calibration import CalibratedClassifierCV
    from sklearn.decomposition import TruncatedSVD
    from sklearn.model_selection import cross_val_predict
else:
    print("Inspection mode: code not executed.")


### 4.2 Supplementary helper functions

Define reusable helpers for fold-safe winsorisation, fixed data splitting and repeated sensitivity evaluation.

In [ ]:
if PORTFOLIO_EXECUTE:
    class FoldWinsorizer(BaseEstimator, TransformerMixin):
        """Training-fold caps for plausible but influential BMI/glucose extremes."""

        def __init__(self, lower=0.005, upper=0.995):
            self.lower = lower
            self.upper = upper
            self.columns = ["bmi", "avg_glucose_level"]

        def fit(self, X, y=None):
            frame = pd.DataFrame(X).copy()
            self.bounds_ = {
                column: (float(frame[column].quantile(self.lower)), float(frame[column].quantile(self.upper)))
                for column in self.columns
            }
            return self

        def transform(self, X):
            frame = pd.DataFrame(X).copy()
            for column, (low, high) in self.bounds_.items():
                frame[column] = frame[column].clip(low, high)
            return frame

    def fixed_split():
        data = pd.read_csv(DATA_PATH)
        data["smoking_status"] = data["smoking_status"].replace("Unknown", np.nan)
        X = data.drop(columns=["stroke", "id"])
        y = data["stroke"].astype(int)
        return train_test_split(
            X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
        )

    def repeated_metrics(name, estimator, X, y, splits=5, repeats=3):
        probabilities, folds, seconds = repeated_oof_probabilities(
            estimator, X, y, splits=splits, repeats=repeats
        )
        threshold, _ = select_f2_threshold(y, probabilities)
        metrics = metric_bundle(y, probabilities, threshold)
        metrics.update({"variant": name, "cv_seconds": seconds})
        folds.insert(0, "variant", name)
        return metrics, folds, probabilities
else:
    print("Inspection mode: code not executed.")


### 4.3 Prepare supplementary analyses

Prepare the fixed development/test split and tuned model configurations for the prespecified sensitivity analyses.

In [ ]:
if PORTFOLIO_EXECUTE:
    X_dev, X_test, y_dev, y_test = fixed_split()

    imbalance_ratio = float((len(y_dev) - y_dev.sum()) / y_dev.sum())

    fitted_xgb = clone(tuned_models["XGBoost"])

    fitted_logistic = clone(tuned_models["Logistic regression"])

    rows = []

    fold_tables = []

    engineered_xgb = clone(fitted_xgb)

    raw_xgb = make_pipeline(
        clone(fitted_xgb.named_steps["model"]), imputation="knn", engineered=False
    )
else:
    print("Inspection mode: code not executed.")


### 4.3.1 Run feature and preprocessing sensitivity analyses

Compare engineered and raw features, dimension reduction, smoking-status handling and fold-safe winsorisation using development-only repeated validation.

In [ ]:
if PORTFOLIO_EXECUTE:
    for name, estimator in [
        ("Engineered XGBoost", engineered_xgb),
        ("Raw-feature XGBoost", raw_xgb),
    ]:
        metrics, folds, _ = repeated_metrics(name, estimator, X_dev, y_dev)
        rows.append({"experiment": "feature_ablation", **metrics})
        fold_tables.append(folds)

    pca_steps = [(name, clone(step)) for name, step in fitted_logistic.steps[:-1]]

    pca_logistic = Pipeline(
        pca_steps
        + [
            ("svd", TruncatedSVD(n_components=12, random_state=RANDOM_STATE)),
            ("model", clone(fitted_logistic.named_steps["model"])),
        ]
    )

    metrics, folds, _ = repeated_metrics("Engineered logistic + SVD(12)", pca_logistic, X_dev, y_dev)

    rows.append({"experiment": "dimension_reduction", **metrics})

    fold_tables.append(folds)

    smoking_mode = clone(fitted_logistic)

    smoking_mode.set_params(preprocess__categorical__imputer__strategy="most_frequent")

    for name, estimator in [
        ("Smoking missing as explicit Unknown", clone(fitted_logistic)),
        ("Smoking missing imputed to mode", smoking_mode),
    ]:
        metrics, folds, _ = repeated_metrics(name, estimator, X_dev, y_dev)
        rows.append({"experiment": "smoking_missingness", **metrics})
        fold_tables.append(folds)

    winsor_steps = [(name, clone(step)) for name, step in fitted_logistic.steps]

    winsorised = Pipeline([("winsor", FoldWinsorizer())] + winsor_steps)

    for name, estimator in [
        ("Clinically plausible extremes retained", clone(fitted_logistic)),
        ("Fold-safe 0.5/99.5% winsorisation", winsorised),
    ]:
        metrics, folds, _ = repeated_metrics(name, estimator, X_dev, y_dev)
        rows.append({"experiment": "outlier_sensitivity", **metrics})
        fold_tables.append(folds)

    unweighted = clone(fitted_logistic).set_params(model__class_weight=None)

    preprocessing_steps = [(name, clone(step)) for name, step in fitted_logistic.steps[:-1]]
else:
    print("Inspection mode: code not executed.")


### 4.3.2 Challenge imbalance handling

Compare no weighting, cost weighting and fold-safe random oversampling.

In [ ]:
if PORTFOLIO_EXECUTE:
    rows = [r for r in rows if r.get("experiment") != "imbalance_strategy"]; fold_tables = [f for f in fold_tables if f.empty or "variant" not in f.columns or str(f["variant"].iloc[0]) not in {"Unweighted logistic", "Cost-weighted logistic", "Random-over-sampled logistic"}]; oversampled = ImbPipeline(
        preprocessing_steps
        + [
            ("oversample", RandomOverSampler(random_state=RANDOM_STATE)),
            (
                "model",
                LogisticRegression(
                    C=fitted_logistic.named_steps["model"].C,
                    max_iter=3000,
                    class_weight=None,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

    for name, estimator in [
        ("Unweighted logistic", unweighted),
        ("Cost-weighted logistic", clone(fitted_logistic)),
        ("Random-over-sampled logistic", oversampled),
    ]:
        metrics, folds, _ = repeated_metrics(name, estimator, X_dev, y_dev)
        rows.append({"experiment": "imbalance_strategy", **metrics})
        fold_tables.append(folds)

    pd.DataFrame(rows).to_csv(TABLE_DIR / "supplementary_development_experiments.csv", index=False)

    pd.concat(fold_tables, ignore_index=True).to_csv(
        TABLE_DIR / "supplementary_repeated_cv_folds.csv", index=False
    )

    outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE + 31)

    calibration_rows = []

    calibration_oof = {}

    for method in ["sigmoid", "isotonic"]:
        calibrated = CalibratedClassifierCV(
            estimator=clone(fitted_xgb), method=method, cv=3, ensemble=False
        )
        probabilities = cross_val_predict(
            calibrated, X_dev, y_dev, cv=outer, method="predict_proba", n_jobs=1
        )[:, 1]
        calibration_oof[method] = probabilities
        threshold, _ = select_f2_threshold(y_dev, probabilities)
        metrics = metric_bundle(y_dev, probabilities, threshold)
        calibration_rows.append({"method": method, **metrics})

    calibration_table = pd.DataFrame(calibration_rows).sort_values(["brier", "pr_auc"])
else:
    print("Inspection mode: code not executed.")


### 4.4 Probability calibration and SHAP interpretability

Select the calibration method using development Brier score, evaluate the calibrated model on the held-out test set, and compute SHAP values for the selected XGBoost model.

In [ ]:
if PORTFOLIO_EXECUTE:
    selected_method = str(calibration_table.iloc[0]["method"])

    calibration_table["selected_on_development_brier"] = calibration_table["method"].eq(selected_method)

    calibration_table.to_csv(TABLE_DIR / "calibration_development_comparison.csv", index=False)

    final_calibrator = CalibratedClassifierCV(
        estimator=clone(fitted_xgb), method=selected_method, cv=5, ensemble=False
    )

    final_calibrator.fit(X_dev, y_dev)

    selected_threshold, threshold_table = select_f2_threshold(
        y_dev, calibration_oof[selected_method]
    )

    test_probabilities = final_calibrator.predict_proba(X_test)[:, 1]

    calibrated_test_metrics = metric_bundle(y_test, test_probabilities, selected_threshold)

    calibrated_intervals = bootstrap_metric_intervals(
        y_test, test_probabilities, selected_threshold
    )

    joblib.dump(final_calibrator, MODEL_DIR / "calibrated_xgboost.joblib")

    threshold_table.assign(model=f"Calibrated XGBoost ({selected_method})").to_csv(
        TABLE_DIR / "calibrated_threshold_sensitivity.csv", index=False
    )

    write_json(
        TABLE_DIR / "calibrated_xgboost_test_summary.json",
        {
            "calibration_method": selected_method,
            "threshold_selected_on_development": selected_threshold,
            "test_metrics": calibrated_test_metrics,
            "test_confidence_intervals": calibrated_intervals,
        },
    )

    fitted_xgb.fit(X_dev, y_dev)

    transform = fitted_xgb[:-1]

    transformed = transform.transform(X_test)

    feature_names = fitted_xgb.named_steps["preprocess"].get_feature_names_out()

    sample_size = min(1500, transformed.shape[0])

    rng = np.random.default_rng(RANDOM_STATE)

    sample_indices = rng.choice(transformed.shape[0], size=sample_size, replace=False)

    sample = transformed[sample_indices]

    if hasattr(sample, "toarray"):
        sample = sample.toarray()

    explainer = shap.TreeExplainer(fitted_xgb.named_steps["model"])

    shap_values = np.asarray(explainer.shap_values(sample))

    if shap_values.ndim == 3:
        shap_values = shap_values[:, :, 1]
else:
    print("Inspection mode: code not executed.")


### 4.4.1 Global SHAP feature importance

Summarise global feature importance using mean absolute SHAP values for the selected XGBoost model.

In [ ]:
if PORTFOLIO_EXECUTE:
    shap_table = pd.DataFrame(
        {"feature": feature_names, "mean_abs_shap": np.abs(shap_values).mean(axis=0)}
    ).sort_values("mean_abs_shap", ascending=False)

    shap_table.to_csv(TABLE_DIR / "shap_global_importance.csv", index=False)

    top = shap_table.head(15).sort_values("mean_abs_shap")

    plt.figure(figsize=(9.2, 6.4))

    plt.barh(top["feature"].str.replace("numeric__", "", regex=False).str.replace("categorical__", "", regex=False), top["mean_abs_shap"], color="#145D72")

    plt.xlabel("Mean absolute SHAP value")

    plt.ylabel("Model input")

    plt.title("Global predictive contribution in the selected XGBoost model")

    plt.tight_layout()

    plt.savefig(FIGURE_DIR / "fig15_shap_global_importance.png", dpi=220, bbox_inches="tight")

    display(Image(filename=str(FIGURE_DIR / "fig15_shap_global_importance.png")))
else:
    print("Inspection mode: code not executed.")


### 4.5 Held-out calibration evaluation

Keep the calibrated curve separate from the raw-model comparison.

In [ ]:
if PORTFOLIO_EXECUTE:
    plt.close()

    bins = pd.qcut(pd.Series(test_probabilities), q=10, duplicates="drop")

    curve = pd.DataFrame(
        {"predicted": test_probabilities, "outcome": y_test.to_numpy(), "bin": bins}
    ).groupby("bin", observed=False).agg(
        mean_predicted=("predicted", "mean"), observed_rate=("outcome", "mean"), n=("outcome", "size")
    )

    curve.to_csv(TABLE_DIR / "calibrated_xgboost_calibration_bins.csv", index=False)

    plt.figure(figsize=(7.2, 6.2))

    plt.plot([0, max(curve.mean_predicted.max(), curve.observed_rate.max())], [0, max(curve.mean_predicted.max(), curve.observed_rate.max())], "--", color="#4B5563", label="Ideal")

    plt.plot(curve.mean_predicted, curve.observed_rate, marker="o", linewidth=2.2, color="#C26D2D", label=f"Calibrated XGBoost ({selected_method})")

    plt.xlabel("Mean predicted probability")

    plt.ylabel("Observed stroke proportion")

    plt.title("Held-out calibration after development-only probability correction")

    plt.legend(frameon=False)

    plt.tight_layout()

    plt.savefig(FIGURE_DIR / "fig16_calibrated_xgboost_curve.png", dpi=220, bbox_inches="tight")

    display(Image(filename=str(FIGURE_DIR / "fig16_calibrated_xgboost_curve.png")))
else:
    print("Inspection mode: code not executed.")


### 4.5.1 Report calibrated held-out metrics

Report the selected calibration method and corresponding held-out test metrics separately from the primary uncalibrated model results.

In [ ]:
if PORTFOLIO_EXECUTE:
    plt.close()

    print(json.dumps({"selected_calibration": selected_method, "test": calibrated_test_metrics}, indent=2))
else:
    print("Inspection mode: code not executed.")


## 5 Results and checks

The next cells display bounded tables matching the report.

### 5.1 Compare preprocessing and models

Check that the selected method follows the stated development-only rule.

In [ ]:
if PORTFOLIO_EXECUTE:
    display(pd.read_csv(TABLE_DIR / 'imputation_comparison_summary.csv').round(4))
    display(pd.read_csv(TABLE_DIR / 'oof_model_metrics.csv').round(4))
else:
    print("Inspection mode: code not executed.")


### 5.2 Review the untouched test

Read PR-AUC, recall, precision and specificity together; accuracy alone is misleading here.

In [ ]:
if PORTFOLIO_EXECUTE:
    display(pd.read_csv(TABLE_DIR / 'untouched_test_metrics.csv').round(4))
else:
    print("Inspection mode: code not executed.")


### 5.3 Review sensitivity and subgroup evidence

Small subgroup event counts limit conclusions even when point estimates differ.

In [ ]:
if PORTFOLIO_EXECUTE:
    display(pd.read_csv(TABLE_DIR / 'supplementary_development_experiments.csv').round(4))
    display(pd.read_csv(TABLE_DIR / 'subgroup_audit.csv').round(4))
else:
    print("Inspection mode: code not executed.")


### 5.4 Confirm the final summary

This dictionary is the machine-readable source for the report's principal numbers.

In [ ]:
if PORTFOLIO_EXECUTE:
    summary = json.loads((TABLE_DIR / 'final_summary.json').read_text(encoding='utf-8'))
    summary
else:
    print("Inspection mode: code not executed.")


## 6 Takeaways

1. Engineered XGBoost led development PR-AUC and remained the prespecified selected model, although logistic regression achieved higher PR-AUC on the untouched test.
2. At the development-selected threshold (0.7494), XGBoost detected 44.6% of stroke-labelled test records at 91.4% specificity, with 8.7% precision in this highly imbalanced dataset.
3. Sigmoid calibration substantially reduced Brier error; using its development-selected threshold, test recall increased to 58.6% while specificity decreased to 88.1%.
4. Subgroup estimates are exploratory and sometimes based on few stroke events; together with dataset provenance and validation limitations, they do not support causal, clinical, or deployment claims.

## 7 Export working evidence

The final cell packages generated evidence and analysis scripts. In Colab it starts a download automatically.

In [ ]:
if PORTFOLIO_EXECUTE:
    zip_path = project_root / 'submission' / 'M32366_CW2_Colab_Evidence.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
        for folder in [project_root / 'evidence']:
            for item in folder.rglob('*'):
                if item.is_file() and not item.name.startswith('_temp_fresh_'):
                    archive.write(item, item.relative_to(project_root))
    print('Evidence ZIP ready: submission/M32366_CW2_Colab_Evidence.zip')
    if IN_COLAB:
        from google.colab import files
        files.download(str(zip_path))
else:
    print("Inspection mode: code not executed.")
